# PolicyPilot held-out evaluation

Use a hosted Colab CPU runtime. The notebook fetches the verified R2 archive into the cloud runtime and compares both networks with the existing baselines on the frozen English synthetic test split.


In [ ]:
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import zipfile

if not os.environ.get("COLAB_RELEASE_TAG") or not Path("/content").is_dir():
    raise RuntimeError("Connect to a hosted Google Colab CPU runtime.")

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Select a Colab runtime that includes PyTorch.")
if importlib.util.find_spec("boto3") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "boto3==1.43.127"], check=True)

workspace = Path(tempfile.mkdtemp(prefix="policypilot-evaluation-", dir="/content"))
encoded_source = (
    '...'
)

source_paths = ['specialists/__init__.py', 'specialists/artifacts.py', 'specialists/contracts.py', 'specialists/execution.py', 'specialists/features.py', 'specialists/models.py', 'specialists/runtime.py', 'specialists/evaluate.py', 'evaluation/specialists-v1/manifest.json', 'evaluation/specialists-v1/test-span.jsonl', 'evaluation/specialists-v1/test-rank.jsonl']
with zipfile.ZipFile(io.BytesIO(base64.b64decode(encoded_source))) as source:
    for name in source_paths:
        path = workspace / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(source.read(name))
sys.path.insert(0, str(workspace))
from specialists.evaluate import ARCHIVE_BYTES, ARCHIVE_SHA256, evaluate, require_hosted_colab
require_hosted_colab()
print("Evaluation source and frozen test data are ready.")


Use an R2 Object Read-only S3 key scoped to the staging bucket. Enter both credentials at the hidden prompts in the next cell. The metrics are printed at the end.


In [ ]:
import getpass
import hashlib
import json
import boto3
from botocore.config import Config

def score_saved_checkpoints():
    require_hosted_colab()
    account = "7e49cfe64d89569b5e5829ccc329ca53"
    bucket = "policypilot-staging"
    key = f"specialists/sha256/{ARCHIVE_SHA256}.tar.gz"
    access = getpass.getpass("R2 read-only S3 Access Key ID: ")
    secret = getpass.getpass("R2 read-only S3 Secret Access Key: ")
    if not access or not secret:
        raise ValueError("Enter both S3 credential values.")
    client = boto3.client(
        "s3", endpoint_url=f"https://{account}.r2.cloudflarestorage.com",
        region_name="auto", aws_access_key_id=access, aws_secret_access_key=secret,
        config=Config(connect_timeout=5, read_timeout=20, retries={"total_max_attempts": 1}),
    )
    try:
        with tempfile.TemporaryDirectory(dir=workspace) as temp:
            archive = Path(temp) / "checkpoint.tar.gz"
            response = client.get_object(Bucket=bucket, Key=key)
            body, digest, total = response["Body"], hashlib.sha256(), 0
            try:
                if response["ContentLength"] != ARCHIVE_BYTES:
                    raise ValueError("R2 object length differs from the backup receipt.")
                with archive.open("xb") as file:
                    while chunk := body.read(256 * 1024):
                        total += len(chunk)
                        if total > ARCHIVE_BYTES:
                            raise ValueError("R2 object exceeds the expected size.")
                        file.write(chunk)
                        digest.update(chunk)
            finally:
                body.close()
            if total != ARCHIVE_BYTES or digest.hexdigest() != ARCHIVE_SHA256:
                raise ValueError("R2 object checksum differs from the backup receipt.")
            return evaluate(archive, workspace / "evaluation/specialists-v1")
    finally:
        client.close()

report = score_saved_checkpoints()
report_path = Path("/content/policypilot-heldout-report.json")
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps({
    "archive_sha256": report["archive_sha256"],
    "split": report["split"],
    "span_exact_f1": {k: v["exact_span"]["f1"] for k,v in report["metrics"]["span"].items()},
    "rank_top1_accuracy": {k: v["group_top1_accuracy"] for k,v in report["metrics"]["rank"].items()},
    "runtime": report["runtime"],
}, indent=2))
print("Full metrics saved to", report_path)
